In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

<div style="background-color:#FFF5E1; padding:30px; border-radius:10px; margin-bottom:30px;">
    <h1 style="color:#0C1844; font-size:40px; font-weight:bold; margin:0;text-align:center;">
        🚀 Shipping Performance Analysis for E-commerce
    </h1> <br><br><br>
<h2 style="color:#0C1844; font-size:24px; margin-top:25px;">Business Problem</h2>
<p style="line-height:1.6; font-size:16px;">
    Delivery delays are a critical pain point in e-commerce, directly impacting customer satisfaction, retention rates, and revenue. For online retailers, understanding the root causes of shipping delays is essential to optimize logistics, improve vendor selection, and enhance operational efficiency.
</p>

<h2 style="color:#0C1844; font-size:24px; margin-top:25px;">Project Focus</h2>
<p style="line-height:1.6; font-size:16px;">
    Using the <strong>Brazilian Olist E-commerce dataset</strong>, this analysis will:
</p>
<ul style="line-height:1.6; font-size:16px; padding-left:20px;">
    <li>Identify the <strong>top predictors</strong> of shipping delays (e.g., seller location, product category, payment methods)</li>
    <li>Quantify the <strong>financial and operational impact</strong> of delays</li>
    <li>Provide <strong>data-driven recommendations</strong> to mitigate delays</li>
</ul>

<h2 style="color:#0C1844; font-size:24px; margin-top:25px;">Why This Matters</h2>
<p style="line-height:1.6; font-size:16px;">
    A 2023 Baymard Institute study found that 24% of cart abandonments are due to slow delivery estimates. Proactively addressing delay factors can significantly improve customer lifetime value (CLV) and reduce churn.
</p>
    <p style="margin:0; font-weight:bold; font-size:18px;">📌 Key Questions:</p>
    <ol style="margin:10px 0 0 0; padding-left:20px; line-height:1.6;">
        <li>Which product categories have the highest delay rates?</li>
        <li>How does seller location impact delivery performance?</li>
        <li>Are certain payment methods correlated with faster/slower shipping?</li>
    </ol>
</div>
</div>

<div style="background-color:#FF6969; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h2 style="color:#FFF5E1; font-size:28px; margin:0;">
        🧩 1. Data Cleaning
    </h2>
</div>


In [ ]:
# Import all the dataset
orders = pd.read_csv("Datasets/olist_orders_dataset.csv")
customers = pd.read_csv('Datasets/olist_customers_dataset.csv')
geolocation = pd.read_csv('Datasets/olist_geolocation_dataset.csv')
order_items = pd.read_csv('Datasets/olist_order_items_dataset.csv')
payments = pd.read_csv('Datasets/olist_order_payments_dataset.csv')
reviews = pd.read_csv('Datasets/olist_order_reviews_dataset.csv')
products = pd.read_csv('Datasets/olist_products_dataset.csv')
sellers = pd.read_csv('Datasets/olist_sellers_dataset.csv')
categories=pd.read_csv('Datasets/product_category_name_translation.csv')

In [ ]:
dfs = {
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "geolocation": geolocation,
    "categories": categories
}
for name, df in dfs.items():
    print(f"\n{name.upper()} - NULL values:")
    print(df.isnull().sum(),'\n')

In [ ]:
for name, df in dfs.items():
    print(f"\n{name.upper()} - Duplicates: {df.duplicated().sum()}")

<div style="background-color:#FF6969; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h3 style="color:#FFF5E1; font-size:24px; margin:0;">
        1.1 Handling Duplicates
    </h3>
</div>

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">There are 261831 duplicated values foudn at Geolocation table.</li>
             <li style="margin-bottom: 6px;">Remove duplicated values to make the table clean and concise also make the process run smoother.</li>
        </ul>
    </div>
    </p>
</div>


In [ ]:
geolocation.duplicated(subset=['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']).value_counts()

In [ ]:
geolocation = geolocation.drop_duplicates()
geolocation.duplicated().sum()

<div style="background-color:#FF6969; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h3 style="color:#FFF5E1; font-size:24px; margin:0;">
         1.2 Handling Missing Data
    </h3>
</div>

<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        Orders Table
    </h4>
</div>

In [ ]:
orders.info()

In [ ]:
cols_to_datetime = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders[cols_to_datetime] = orders[cols_to_datetime].apply(pd.to_datetime)
orders.dtypes

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">It seems there are many orders without customers delivery date. Next we will take a look at what kinds of orders are lacking this information. </li>      
        </ul>
    </div>
    </p>
</div>

In [ ]:
null_delivery = orders['order_delivered_customer_date'].isnull()
orders[null_delivery].head()

In [ ]:
orders[null_delivery]['order_status'].value_counts()

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
       <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">Some of the orders don't have <b>order_delivered_customer_date</b> because they were either approved,shipped or processed so it's reasonable from logistics standpoint.</li>
            <li style="margin-bottom: 6px;">The same applies for 'Cancelled' and 'Unavailable' orders. </li>
            <li style="margin-bottom: 6px;">However There are 8 orders without <b>order_delivered_customer_date</b> but marked as 'delivered', which make these records unreliable. So we have to remove those records</li>
        </ul>
    </div>
    </p>
</div>

In [ ]:
delivered_order_null_date = orders[null_delivery & (orders["order_status"] == "delivered")]["order_id"].values
delivered_order_null_date

In [ ]:
# Create a copy DF where we don't include the order_id with null order date
# the copy() to signal that we copied the data from the original one so later on Python won't give us any error 
# when we add new columns 
orders = orders[~orders["order_id"].isin(list(delivered_order_null_date))].copy()
orders.isnull().sum()

In [ ]:
# Check the order where order approval date is null
orders[orders["order_approved_at"].isnull()]["order_status"].value_counts()

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
       <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">There are also orders who are not approved but already delivered.
Which meant they don't have to be approved.
So we include the order purchase timestamp as their approval time instead.</li>
            <li style="margin-bottom: 6px;"><b>Delivered_carrier_date</b> can also be null as it is assumed the site will deliver directly to the customers without delivering it to the carrier.</li>
            <li style="margin-bottom: 6px;">When the order is progressing like with the order status = progress, shipped, or cancelled, or unavailable, it makes sense that the <b>order_delivered_customer_date</b> can be NULL</li>
        </ul>
    </div>
    </p>
</div>

In [ ]:
orders.loc[orders['order_approved_at'].isnull() & (orders['order_status'] == 'delivered'), 'order_approved_at'] = orders['order_purchase_timestamp']

In [ ]:
orders.isnull().sum()

In [ ]:
orders['order_approved_at'] = orders['order_approved_at'].fillna(orders['order_purchase_timestamp'])
orders.isna().sum()

<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        Reviews Table
    </h4>
</div>

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">As from the initial exploration, we know that <b>review_comment_title</b> has 87656 null values. 
This field doesn't contribute to our analysis so we will drop it</li>
        </ul>
        </ul>
    </div>
    </p>
</div>

In [ ]:
reviews = reviews.drop('review_comment_title',axis=1)
reviews.columns

<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        Products Table
    </h4>
</div>

In [ ]:
products = products.drop(['product_name_lenght', 'product_description_lenght', 'product_photos_qty'], axis=1)

products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_weight_g'] = products['product_weight_g'].fillna(products['product_weight_g'].median())
products['product_length_cm'] = products['product_length_cm'].fillna(products['product_length_cm'].median())
products['product_height_cm'] = products['product_height_cm'].fillna(products['product_height_cm'].median())
products['product_width_cm'] = products['product_width_cm'].fillna(products['product_width_cm'].median())

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Actions Summary:</h2>
    <ul style="font-size:16px;">
        <li>The records at the orders table have a lot of null values for approval timestamp or customer delivery date timestamp. However as it is investigated, many records have the statuses in between the creation of orders and the delivery time, hence, it is logical that the columns are null.</li>
        <li>Geolocation data has many duplications for locations which we reduced the duplication to transform it to a viable master data table.</li>
    </ul>
</div>


<div style="background-color:#FF6969; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h2 style="color:#FFF5E1; font-size:28px; margin:0;">
        🤝 2. Data Merging
    </h2>
</div>


In [ ]:
orders = orders.merge(customers,how='left',on='customer_id')

In [ ]:
order_items.head()

In [ ]:
orders = orders.merge(order_items[['order_id','product_id','seller_id','price','freight_value']],how='left',on='order_id')

In [ ]:
orders = orders.merge(products[['product_id', 'product_category_name']], how='left', on='product_id')

In [ ]:
orders = orders.merge(categories,on='product_category_name')

In [ ]:
orders.isna().sum()

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">In this analysis, I won't look at payment type so the payment table iss not merged with orders table.

</li>
        </ul>
        </ul>
    </div>
    </p>
</div>

In [ ]:
orders = orders.merge(reviews[['review_id','order_id','review_score','review_creation_date']],how='left',on='order_id')

In [ ]:
geolocation.value_counts()

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">It seems one city can have multiple geolocation_lat and geolocation_lng, so we need to clean up this information before merging the table.

</li>
        </ul>
        </ul>
    </div>
    </p>
</div>



In [ ]:
geo_avg = geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

orders = orders.merge(geo_avg, how='left', left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix')

orders.isna().sum()

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">It seems we have a significant number of cities which don't have geolocation_lat and geolocation_lng. So we have to find median values of each city to fill up the NA values of those locations.
</li>
        </ul>
        </ul>
    </div>
    </p>
</div>


In [ ]:
# use transform to preserve the original dataframe shape so we can easy fillin na later.
median_coords = orders.groupby('customer_city')[['geolocation_lat','geolocation_lng']].transform('median')
orders[['geolocation_lat', 'geolocation_lng']] = orders[['geolocation_lat', 'geolocation_lng']].fillna(median_coords)
orders.isna().sum()

In [ ]:
orders.loc[orders['geolocation_lng'].isna()]['customer_city'].head()

In [ ]:
orders.loc[(orders['customer_city']=='alto sao joao') | (orders['customer_city']=='domiciano ribeiro')][['customer_city','geolocation_lat','geolocation_lng']]

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">It seems that some of geolocation_lat and geolocation_lng still have some NAs. It can be speculated that if the customer city doesn't have no non-NA values then the their median will still be NA
</li>
            <li style="margin-bottom: 6px;">To fix this issue, we can use the median long and lat of all cities to fill in NA.
</li>
        </ul>
        </ul>
    </div>
    </p>
</div>

In [ ]:
median_lat = orders['geolocation_lat'].median()
median_long = orders['geolocation_lng'].median()
orders['geolocation_lat'] = orders['geolocation_lat'].fillna(median_lat)
orders['geolocation_lng'] = orders['geolocation_lng'].fillna(median_long)
orders[['geolocation_lat','geolocation_lng']].isna().sum()

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Actions Summary:</h2>
    <ul style="font-size:16px;">
        <li>We merged all the relevant table into one so that we can easily retrieve the data as we need</li>
        <li>Geolocation data has many NA values due to lack of coordination for some cities so we fill in with median values.</li>
    </ul>
</div>


<div style="background-color:#FF6969; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h2 style="color:#FFF5E1; font-size:28px; margin:0;">
        🔍 3. Exploratory Data Analysis (EDA)
    </h2>
</div>


<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
       3.1. How many orders are delayed?
    </h4>
</div>

In [ ]:
orders['delay_period'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['delayed']= orders['delay_period'] >0
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()

In [ ]:
delivered_orders.head()

In [ ]:
delivered_orders.duplicated().sum()

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">Going back to Order table, we can see that there is no product quantity. Given the amount of duplication in the order table in all columns, we could say that each item is logged into the table every time it is purchased as single line of row. For example, if the quantity of product A is 3, there are 3 rows of product A with the same detail in the order table.
</li>
            <li style="margin-bottom: 6px;">So we won't remove the duplication on this table.
</li>
        </ul>
        </ul>
    </div>
    </p>
</div>

In [ ]:
delayed_orders = delivered_orders['delayed'].value_counts(normalize=True)
delayed_orders = delayed_orders.mul(100)
delayed_orders

In [ ]:
plt.figure(figsize=(8, 6))

# Create bar plot - convert series to DataFrame for seaborn
plot_data = delayed_orders.reset_index()
plot_data.columns = ['status', 'percentage']
ax = sns.barplot(data=plot_data, 
                 x='status', 
                 y='percentage',
                 palette='Reds_r')

# Customize labels
ax.set_xticklabels(['On Time', 'Delayed'])
ax.set_ylabel('Percentage of Orders', fontsize=12)
ax.set_xlabel('Delivery Status', fontsize=12)
plt.title('Delivery Performance', fontsize=14, pad=20)

# Add value labels on bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', 
               (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='center', 
               xytext=(0, 10), 
               textcoords='offset points',
               fontsize=12)

# Improve styling
sns.despine()
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, 100)  # Set y-axis to 0-100% scale

plt.tight_layout()
plt.show()

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Findings:</h2>
    <ul style="font-size:16px;">
        <li>Only a very small percentage of the orders are delayed. We could suspect this is either due to efficient logistic shipping system or that the estimated delivery date for customers are very far away from the time it is created.</li>
    </ul>
</div>


<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        3.2. Which states or cities have the highest delayed rates?
    </h4>
</div>

In [ ]:
top_delayed_city = delivered_orders.loc[delivered_orders['delayed']==True]['customer_city'].value_counts().head(20).reset_index()
top_delayed_city.columns = ['city','delay_count']
top_delayed_city.head()

In [ ]:
def plot_barh_chart(df, x, y, xlabel, ylabel, title, palette='Reds_r'):
    plt.figure(figsize=(10, 8))
    sns.barplot(data=df, x=x, y=y, palette=palette)  # Uses default or custom palette
    plt.title(title, fontsize=15, weight='bold', pad=20)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_barh_chart(top_delayed_city,'delay_count','city','Delay Count','Customer Cities','Top Cities with Highest number of delays')

In [ ]:
delivered_orders['delayed'] = delivered_orders['delayed'].astype(bool)

In [ ]:
delivered_orders = delivered_orders.groupby('customer_city').filter(lambda x: x['order_id'].count() > 0)

In [ ]:
# Create a new DF called city_stats to group customer_city and see its total orders and delayed_orders
# use assign() to create a new column while creating a new DF
city_stats = (
    delivered_orders.groupby('customer_city')
    .agg(
        total_orders=('order_id', 'count'),
        delayed_orders=('delayed', 'sum')  # Assuming 'delayed' is boolean
    )
    .assign(delay_pct=lambda x: (x['delayed_orders'] / x['total_orders'] * 100).round(2))
    .reset_index()
    .sort_values('total_orders', ascending=False)
)

In [ ]:
city_stats.head()

In [ ]:
plt.figure(figsize=(12, 8))
scatter = sns.scatterplot(
    data=city_stats,
    x='total_orders',
    y='delay_pct',
    hue='delay_pct',  # Color by delay rate
    size='total_orders',  # Size by order volume
    palette='Reds_r',    
    alpha=0.7,
    sizes=(20, 200)      # Min/max bubble size
)

# Highlight key cities (top 5 by order volume)
# '_' to ignore the index, only get the information of the row
top_cities = city_stats.head(5)
for _, row in top_cities.iterrows():
    plt.text(
        # Coordinate x for the label
        row['total_orders'] * 1.05,  # Offset x-position
        # Coordinate y for the label
        row['delay_pct'],
        # Label to add
        row['customer_city'],
        fontsize=9,
        #horizontal align = left
        ha='left'
    )

plt.title('Delivery Delay Rate vs. Order Volume by City', fontsize=14)
plt.xlabel('Total Orders (Log Scale)')  # Use log scale if skewed
plt.ylabel('Delay Rate (%)')
plt.yscale('linear')
plt.xscale('log')  # Helps if order counts vary widely
plt.grid(True, linestyle='--', alpha=0.3)

# Add correlation line
sns.regplot(
    data=city_stats,
    x='total_orders',
    y='delay_pct',
    scatter=False,
    color='gray',
    line_kws={'alpha': 0.5}
)

plt.tight_layout()
plt.show()

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Findings:</h2>
    <ul style="font-size:16px;">
         <li style="margin-bottom: 6px;">Just look at the total number of delays per city doesn't give you a full picture. The delay count might be high but then it might be due to the total number of orders.</li>
            <li style="margin-bottom: 6px;">Using scatter-plot it can be seen that delayed rates are high at the cities with small volume of orders.
                The higher the volume of orders, the less likely it will be cancelled.
</li>
        <li style="margin-bottom: 6px;">We also had a lot of cities with very little delays that is almost 0%.
</li>
    </ul>
</div>


<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        3.3. Do the delivery days and delays rate vary among product categories?
    </h4>
</div>

In [ ]:
top_delayed_category = delivered_orders.groupby('product_category_name_english')['delayed'].mean().sort_values(ascending=False).head(10)
top_delayed_category = top_delayed_category.reset_index() #to turn Series into DataFrame
top_delayed_category = top_delayed_category.sort_values(by='delayed',ascending=False)
top_delayed_category

In [ ]:
plot_barh_chart(top_delayed_category,'delayed','product_category_name_english','Delay Percentage','Product Category',
                'Top product categories with highest delayed rates')

In [ ]:
# Calculate the time since the order is purchased until it is delivered
delivered_orders['total_time']= (delivered_orders['order_delivered_customer_date']-delivered_orders['order_purchase_timestamp']).dt.days

In [ ]:
slowest_delivery_categories = delivered_orders.groupby('product_category_name_english')['total_time'].median().nlargest(10)
fastest_delivery_categories = delivered_orders.groupby('product_category_name_english')['total_time'].median().nsmallest(10)

In [ ]:
slowest_delivery_categories = slowest_delivery_categories.reset_index()
slowest_delivery_categories.columns=['product category','delivery days']

In [ ]:
plot_barh_chart(slowest_delivery_categories,'delivery days','product category','Delivery Days','Product Category',
                'Top 10 slowest delivery product categories')

In [ ]:
fastest_delivery_categories = fastest_delivery_categories.reset_index()
fastest_delivery_categories.columns=['product category','delivery days']

In [ ]:
plot_barh_chart(fastest_delivery_categories,'delivery days','product category','Delivery Days','Product Category',
                'Top 10 fastest delivery product categories','viridis')

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Findings:</h2>
    <ul style="font-size:16px;">
         <li style="margin-bottom: 6px;">Furniture/mattres and upholstery, home_comfor_2, and audio are the categories with the highest number of delays, however the highest percentage is only 12% which is not a big number.</li>
        <li style="margin-bottom: 6px;">When looking at total delivery days, office furnitures take more than 17 days from order created date until it reaches customer hand. Products in security and services group also come close at second place with total of 15 days.
</li>
        <li style="margin-bottom: 6px;">Top fatest delivered groups have the total delivery of less than 8 days, which are much shorter than the slowest delivery group. Arts and craftmanship is delivered the fastest with less than 5 days. Fashion children clothes also took less than a week to be delivered.
</li>
    </ul>
</div>

<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        3.4. How does average delays change over time?
    </h4>
</div>

In [ ]:
delivered_orders['order_month'] = delivered_orders['order_purchase_timestamp'].dt.to_period('M')
monthly_delay = delivered_orders.groupby('order_month')['delay_period'].mean().round()
monthly_delay = monthly_delay.reset_index()
monthly_delay.columns = ['order_month','average_delay_days']
monthly_delay.head()

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(
    data=monthly_delay, 
    x='order_month', y='average_delay_days', palette='viridis')
plt.title('Average Delays (in Days) per Month')
plt.xlabel('Month')
plt.ylabel('Average Delay Rate')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">It seems only in 2016 (2016-09) that most of the delivery were delayed.
From that point, the average delivery rate was negative which means in most cases, the customers received the orders before estimated time.</li>
            <li style="margin-bottom: 6px;">We could dig in further to see what happened during that time frame compared to the rest.</li>
        </ul>
        </ul>
    </div>
    </p>
</div>

In [ ]:
delivered_orders[delivered_orders['order_month']=='2016-09']

<div style="background-color:#FFF5E1; padding:15px 25px; border-left:5px solid #0C1844; margin:20px 0; border-radius:8px;">
    <p style="font-size:16px; color:#0C1844; margin:0;">
        <ul style="
            margin: 12px 0;
            padding-left: 20px;
        ">
            <li style="margin-bottom: 6px;">All the delayed orders from 2016-09 came from one order with a very high delay period (36 days) for all 3 itimes which made the total delivery time from the time order is purchased 54 days.</li>
            <li style="margin-bottom: 6px;">Since the data is quite skewed, we can say deduct that this case is an outlier for the time period.</li>
        </ul>
        </ul>
    </div>
    </p>
</div>

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Findings:</h2>
    <ul style="font-size:16px;">
         <li style="margin-bottom: 6px;">The e-commerce site from 2016 onwards have always delivered faster than it is estimated to reach the customers. Only one month at the beginning that they had delayed orders which originated from one customer and one order only. From this we can conclude that the site on average always deliver faster than they promised.</li>
    </ul>
</div>

<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        3.5. What is the average processing days between each stage of the order-to-delivered process?
    </h4>
</div>

In [ ]:
orders.columns

In [ ]:
orders['approval_time'] = orders['order_approved_at'] - orders['order_purchase_timestamp']
orders['waiting_time'] = orders['order_delivered_carrier_date'] - orders['order_approved_at']
orders['shipping_time'] = orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']
orders['total_time']= orders['order_delivered_customer_date']- orders['order_purchase_timestamp']
orders['estimated_time'] = orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']

In [ ]:
time_columns = ['approval_time', 'waiting_time', 'shipping_time', 'total_time', 'estimated_time']

#pd.Timedelta(0) explicitly compares time durations (more reliable than <0 for edge cases)
for col in time_columns:
    negatives = orders[orders[col] < pd.Timedelta(0)]
    print(f'{col}: {len(negatives)} negative values')

In [ ]:
orders = orders[orders['waiting_time'] >= pd.Timedelta(0)]
orders = orders[orders['shipping_time'] >= pd.Timedelta(0)]

In [ ]:
time_columns = ['approval_time', 'waiting_time', 'shipping_time', 'total_time', 'estimated_time']

summary = pd.DataFrame({
    'mean': orders[time_columns].mean(),
    'median': orders[time_columns].median(),
    'std': orders[time_columns].std()
})

summary = summary.applymap(lambda x: pd.to_timedelta(x.round('s')))

summary

In [ ]:
summary = pd.DataFrame(
    {
        'median': [
            orders['approval_time'].median(),
            orders[orders['waiting_time'].notna()]['waiting_time'].median(),
            orders[orders['shipping_time'].notna()]['shipping_time'].median(),
            orders['total_time'].median(),
            orders['estimated_time'].median()
            
        ],
        'std':[
            orders['approval_time'].std(),
        orders[orders['waiting_time'].notna()]['waiting_time'].std(),
        orders[orders['shipping_time'].notna()]['shipping_time'].std(),
        orders['total_time'].std(),
        orders['estimated_time'].std()
        ]
    }, index=['Approval', 'Waiting', 'Shipping', 'Total', 'Estimated']
)
summary

In [ ]:
# apply to_timedelta to all rows to convert it to seconds accurate
summary = summary.applymap(lambda x: pd.to_timedelta(x.round('s')))
summary

In [ ]:
summary['median_hours'] = summary['median'].dt.total_seconds() / 3600
summary['std_hours'] = summary['std'].dt.total_seconds() / 3600
summary['label'] = summary['median_hours'].apply(lambda h: f"{h:.0f}h\n({h/24:.1f}d)")
summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(summary.index, summary['median_hours'], yerr=summary['std_hours'],
             fmt='-o', color='#4C72B0', capsize=6, lw=2, markerfacecolor='white')

for i, (x, y) in enumerate(zip(summary.index, summary['median_hours'])):
    plt.text(x, y + 25, summary['label'].iloc[i], ha='right', fontsize=10, color='#333')

plt.title('Median Delivery Time per Stage (with Std Dev)', fontsize=14, weight='bold')
plt.ylabel('Time (hours)', fontsize=12)
plt.ylim(0, 800)
plt.grid(True)
plt.tight_layout()

for spine in ['top', 'right']:
    plt.gca().spines[spine].set_visible(False)

plt.show()

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Findings:</h2>
    <ul style="font-size:16px;">
         <li style="margin-bottom: 6px;"><b>Approval Time:</b> The median time for Approval period is 20 minutes which means most of the orders are approved instantly and quick within the first hour of order create. This reflects the efficiency of the front-end system which allows users to approve quickly.</li>
         <li style="margin-bottom: 6px;"><b>Waiting Time:</b> Waiting time median is 1.9 days but the standard deviation is 84 hours which is quite high. This shows the difference between each seller when it comes to pick up the products from the warehouse. Some sellers might react swiftly, some might react slowly.</li>
        <li style="margin-bottom: 6px;"><b>Shipping Time:</b> The most dominant and time consuming stage. The median is 7 days but standard deviation can range up to 8 days (200 hours). This can happen due to the regional access and transporation methods. This section is the most potential bottleneck that we could focus on to resolve total delivery times.</li>
        <li style="margin-bottom: 6px;"><b>Total Time:</b> The total delivery time for customers have median of 10 days with standard deviation of 9 days. There is room for improvement as the e-commerce only focuses on domestic market and not international market thus the total delivery time could be shorter to match the overal international standards.</li>
        <li style="margin-bottom: 6px;"><b>Estimated Time:</b> From previous questions, we suspected the estimated time be set much further than actual delivery time becase we have very little delays from 2016/09 onwards. From the graph, we can see that estimated time is set almost double the actual shipping time.</li>
    </ul>
</div>

<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        3.6. How do size of an order affect the delivery length?
    </h4>
</div>

In [ ]:
order_item_counts = orders.groupby('order_id')['product_id'].count().rename('No_of_items')
order_item_counts.value_counts().nlargest(7)

In [ ]:
orders_with_items = orders.merge(order_item_counts, on='order_id')

In [ ]:
plt.figure(figsize=(12, 10))
sns.boxplot(x='No_of_items', y='delay_period', palette='viridis', data=orders_with_items[(orders_with_items['No_of_items'] <= 10)&(orders_with_items['No_of_items']>0)])
plt.title('Delivery Delay by Number of Items in Order')
plt.xlabel('Number of Items')
plt.ylabel('Delivery Delay (days)');

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Findings:</h2>
    <ul style="font-size:16px;">
         <li style="margin-bottom: 6px;">Median delivery delays time are consistenly below 0 day for all of the number of items.</li>
        <li style="margin-bottom: 6px;"><b>Long Tails (Outliers) happened more at lower number of items</b> Delays happen more offen if the number of items in an order is smaller than 3.</li>
        <li style="margin-bottom: 6px;"><b>Small spread in Each Bucket</b> The IQRs are reasonably small in all groups but it seems order with 9 items got delivered with much more variable in delivery days than others but still early. </li>
        <li style="margin-bottom: 6px;"><b>✅ Business Insight:</b> Small orders have higher probability of being delayed than others.</li>
    </ul>
</div>

<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        3.7. How efficient shipping is relative to product cost
    </h4>
</div>

In [ ]:
orders['freight_ratio']= (orders['freight_value']/orders['price']).round(2)
orders['freight_ratio'].head()

In [ ]:
orders['freight_ratio'].describe()

In [ ]:
avg_freight_ratio = orders.groupby('order_id')['freight_ratio'].mean().reset_index()

In [ ]:
merged_df = orders[['order_id','total_time']].merge(avg_freight_ratio, on='order_id', how='inner')
merged_df

In [ ]:
merged_df['freight_bucket'] = pd.cut(
    merged_df['freight_ratio'],
    bins=[0, 0.13, 0.23, 0.39, 1, 5, 30],
    labels=['Very Low (0-0.13)', 'Low (0.13-0.23)', 'Medium (0.23-0.39)', 'High (0.39-1)', 'Very High (1-5)', 'Extreme (5+)']
)

In [ ]:
# convert the total time back to days
merged_df['delivery_days'] = merged_df['total_time'].dt.total_seconds() / (24 * 3600)

#clear the extreme values
filtered_df = merged_df[merged_df['delivery_days'].between(0, 40)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=filtered_df, x='freight_bucket', y='delivery_days', palette='Reds')

plt.title('📦 Delivery Time by Freight Ratio Bucket', fontsize=16)
plt.xlabel('Freight Ratio Bucket', fontsize=12)
plt.ylabel('Delivery Time (Days)', fontsize=12)
plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

<div style="background-color:#FFF5E1; padding:20px 30px; 
    border-left: 5px groove #C80036;
    border-radius:10px; margin:40px 0;">
    <h2 style="font-size:24px;">📌 Findings:</h2>
    <ul style="font-size:16px;">
         <li style="margin-bottom: 6px;">Median delivery time increases slightly as freight ratio rises. For example, Very Low: Median ~8–9 days. High to Very High: Median ~11–13 days. </li>
         <li style="margin-bottom: 6px;">Intuitively, one might expect higher freight = faster delivery. But here, higher freight ratios correlate with slightly slower or more variable deliveries.</li>
        <li style="margin-bottom: 6px;"><b>Long Tails (Outliers) Persist Across All Buckets</b> All categories have high-end outliers, indicating that delays can happen at any freight level.</li>
        <li style="margin-bottom: 6px;"><b>Moderate Spread in Each Bucket</b> The IQRs are reasonably wide in all groups (roughly 5–15 days), reflecting inconsistency in delivery performance regardless of cost.</li>
        <li style="margin-bottom: 6px;"><b>✅ Business Insight:</b> “Paying more in freight doesn’t guarantee faster delivery.”
This could inform pricing, shipping, or customer communication strategies.</li>
    </ul>
</div>

<div style="background-color:#0C1844; text-align:center; padding:20px; border-radius:10px; margin:40px 0;">
    <h4 style="color:#FFF5E1; font-size:20px; margin:0;">
        3.8. How does reviews score different based on delivery status?
</div>

In [ ]:
orders.review_score.value_counts()

In [ ]:
plt.figure(figsize=(8,5), dpi=120)
sns.countplot(x=orders['review_score'], palette='spring')
plt.title('Distribution of scores', fontsize=17, fontweight='bold', pad=7);